# Batch sample review

Reviews a batch of experiments together, before MERlin decoding (no
barcode/cell analysis). For each experiment it checks that the
acquisition-time FOV analysis (`fov_scheduler.ipynb`) exists and fills in what
is missing, then compares per-round intensity and saturation across the
batch.

## 1 — Setup

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd

MERCI_DIR = Path(os.getcwd()).parent.parent   # MERci/ (notebook lives in MERci/notebooks/after_imaging/)
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.config       import ExperimentConfig
from MERci.common.metadata     import ExperimentMetadata
from MERci.common.experiment_info import load_experiment_info
from MERci.progress            import ProgressTracker
from MERci.analysis.batch_sample_review import backfill_pending, load_all_stats
from MERci.plots.batch_sample_review_plots import plot_round_comparison

## 2 — Select the batch

The `experiment_info.yaml` of each experiment to review (written by
`create_experiment_info.ipynb`).

In [ ]:
EXPERIMENT_INFO_PATHS = [
    # Path(r"D:\experiments\LT048_sample_26\metadata\experiment_info.yaml"),
    # Path(r"D:\experiments\LT048_sample_27\metadata\experiment_info.yaml"),
]

IMAGE_SUFFIX = ".zarr"   # same as the experiments were acquired with

if not EXPERIMENT_INFO_PATHS:
    raise ValueError("Set EXPERIMENT_INFO_PATHS to at least one experiment_info.yaml path.")
print(f"{len(EXPERIMENT_INFO_PATHS)} experiment(s) selected.")

## 3 — Build config/metadata/tracker per experiment

In [ ]:
samples = []
for info_path in EXPERIMENT_INFO_PATHS:
    info_path  = Path(info_path)
    sample_dir = info_path.parent.parent   # metadata/experiment_info.yaml -> SAMPLE_DIR
    info       = load_experiment_info(info_path)

    config = ExperimentConfig.from_sample_dir(
        sample_dir,
        positions_txt  = sample_dir / "positions" / f"positions_{info.sample_name}.txt",
        image_suffix   = IMAGE_SUFFIX,
    )
    meta    = ExperimentMetadata.load(config.round_info_csv, config.positions_txt,
                                       config.data_dir, image_suffix=config.image_suffix)
    tracker = ProgressTracker(config.analysis_dir)

    samples.append({"info": info, "sample_dir": sample_dir, "config": config,
                    "meta": meta, "tracker": tracker})
    print(f"{info.sample_name:20s}  rounds={meta.n_rounds}  fovs={meta.n_fovs}  dir={sample_dir}")

## 4 — Verify analysis is available, backfill anything missing

Usually nothing to do, since analysis runs during acquisition. Pending files
are analysed here in parallel (`config.resolved_n_workers` workers).

In [ ]:
for sample in samples:
    n_backfilled = backfill_pending(sample)
    smry = sample["tracker"].summary(sample["meta"])
    print(f"{sample['info'].sample_name:20s}  "
          f"backfilled={n_backfilled:4d}  "
          f"fovs_done={smry['files_fov_done']}/{smry['files_total']}  "
          f"rounds_done={smry['rounds_done']}/{smry['rounds_total']}")

## 5 — Load stats

All stats CSVs of every sample in one DataFrame (with a `sample_name`
column).

In [ ]:
all_stats = pd.concat([load_all_stats(s) for s in samples], ignore_index=True)
print(f"Loaded {len(all_stats)} (sample x round x fov x frame) stat rows.")

## 6 — Per-round intensity / saturation comparison across the batch

One violin per sample, one figure per round: `median` intensity, and `max`/`p99`
as a saturation proxy (there is no exact saturated-pixel fraction).

In [ ]:
if all_stats.empty:
    print("No stats to plot.")
else:
    for rid in sorted(all_stats["round_id"].unique()):
        plot_round_comparison(all_stats, rid, metric="median")
        plot_round_comparison(all_stats, rid, metric="max")